# CS-781 Phase 3: ensemble of the RR runs (CPU only, ~1 minute)

Attach the output folders of your `Phase3_RR_Train` runs as Kaggle inputs (each has `probs_dev.json` and `probs_test.json`).
This notebook
1. scores every run on dev (macro-F1 over the 13 labels, CL and IT scored together exactly like Codabench),
2. builds a **greedy forward-selection ensemble** (Caruana et al., 2004): repeatedly adds the run (repeats allowed) whose averaged probabilities raise dev macro-F1 most,
3. tunes per-class logit biases on dev and checks with a 2-fold split whether that gain holds on unseen documents,
4. writes `submission_ensemble.zip` plus a uniform-average version for comparison.

Selecting runs and biases on dev overfits dev a little, so trust the *honest* number more than the tuned one.

In [ ]:
import json, glob, random, zipfile
from pathlib import Path
import numpy as np

SEARCH = ["/kaggle/input", "/kaggle/working", "."]
MAX_ROUNDS = 12
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

dirs = sorted({str(Path(p).parent) for r in SEARCH for p in glob.glob(f"{r}/**/probs_dev.json", recursive=True)
               if (Path(p).parent / "probs_test.json").exists()})
runs = []
for d in dirs:
    jd, jt = json.load(open(Path(d) / "probs_dev.json")), json.load(open(Path(d) / "probs_test.json"))
    runs.append(dict(dir=d, name=jd["run"], labels=jd["labels"], dev=jd["docs"], test=jt["docs"]))
assert runs, "no run folders found - attach the outputs of Phase3_RR_Train"
names = [r["name"] for r in runs]
assert len(set(names)) == len(names), f"two inputs contain the same run name: {names}"
LABELS = runs[0]["labels"]; C = len(LABELS)
for r in runs:
    assert r["labels"] == LABELS, f"label order differs in {r['dir']}"
    assert set(r["dev"]) == set(runs[0]["dev"]) and set(r["test"]) == set(runs[0]["test"]), f"document IDs differ in {r['dir']}"
DEV_IDS = sorted(runs[0]["dev"]); TEST_IDS = list(runs[0]["test"])
GOLD = [np.asarray(runs[0]["dev"][k]["gold"]) for k in DEV_IDS]
DOM = [runs[0]["dev"][k]["domain"] for k in DEV_IDS]
P_DEV = [[np.exp(np.asarray(r["dev"][k]["logp"], dtype=np.float64)) for k in DEV_IDS] for r in runs]
P_TEST = [{k: np.exp(np.asarray(r["test"][k]["logp"], dtype=np.float64)) for k in TEST_IDS} for r in runs]
print(len(runs), "runs:", names)

In [ ]:
def macro_f1(y, p):
    cm = np.bincount(y * C + p, minlength=C * C).reshape(C, C)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    P = np.divide(tp, tp + fp, out=np.zeros(C), where=(tp + fp) > 0)
    R = np.divide(tp, tp + fn, out=np.zeros(C), where=(tp + fn) > 0)
    return float(np.divide(2 * P * R, P + R, out=np.zeros(C), where=(P + R) > 0).mean())

def score(prob_docs, bias=None, idx=None):
    idx = range(len(DEV_IDS)) if idx is None else idx
    lp = np.concatenate([np.log(prob_docs[i] + 1e-9) for i in idx])
    if bias is not None:
        lp = lp + bias
    return macro_f1(np.concatenate([GOLD[i] for i in idx]), lp.argmax(1))

def avg(members, P):
    return [sum(P[m][i] for m in members) / len(members) for i in range(len(P[0]))]

def tune_bias(prob_docs, idx, passes=3):
    grid = np.round(np.arange(-3.0, 3.01, 0.25), 2); b = np.zeros(C); best = score(prob_docs, b, idx)
    for _ in range(passes):
        imp = False
        for c in range(C):
            for g in grid:
                b2 = b.copy(); b2[c] = g; f = score(prob_docs, b2, idx)
                if f > best + 1e-9: best, b, imp = f, b2, True
        if not imp: break
    return b, best

print(f"{'run':>36} {'dev macro-F1':>12} {'CL':>7} {'IT':>7}")
cl = [i for i, d in enumerate(DOM) if d == "CL"]; it = [i for i, d in enumerate(DOM) if d == "IT"]
for m, r in enumerate(runs):
    print(f"{r['name']:>36} {score(P_DEV[m]):12.4f} {score(P_DEV[m], idx=cl):7.4f} {score(P_DEV[m], idx=it):7.4f}")

In [ ]:
# ---- greedy forward selection with replacement
members, best = [], -1.0
for _ in range(MAX_ROUNDS):
    cand = [(score(avg(members + [m], P_DEV)), m) for m in range(len(runs))]
    f, m = max(cand)
    if f <= best + 1e-6:
        break
    members.append(m); best = f
    print(f"+ {names[m]:>36} -> dev macro-F1 {f:.4f}")
print("selected (with repeats):", [names[m] for m in members])
uniform = list(range(len(runs)))
print(f"uniform average of all runs: {score(avg(uniform, P_DEV)):.4f} | greedy: {best:.4f}")

ens_dev = avg(members, P_DEV)
all_idx = list(range(len(DEV_IDS)))
BIAS, f_b = tune_bias(ens_dev, all_idx)
order = all_idx[:]; random.Random(0).shuffle(order); h = len(order) // 2
before, after = [], []
for a, b in ((order[:h], order[h:]), (order[h:], order[:h])):
    bb, _ = tune_bias(ens_dev, a)
    before.append(score(ens_dev, None, b)); after.append(score(ens_dev, bb, b))
h0, h1 = float(np.mean(before)), float(np.mean(after))
USE_BIAS = h1 > h0
print(f"bias tuning on dev: {best:.4f} -> {f_b:.4f} | honest 2-fold: {h0:.4f} -> {h1:.4f} | use bias: {USE_BIAS}")

In [ ]:
def write(prob_test, bias, name):
    out = {}
    for k in TEST_IDS:
        lp = np.log(prob_test[k] + 1e-9) + (0 if bias is None else bias)
        out[k] = [LABELS[i] for i in lp.argmax(1)]
        assert len(out[k]) == runs[0]["test"][k]["n"]
    jp, zp = OUT / f"prediction_{name}.json", OUT / f"submission_{name}.zip"
    json.dump(out, open(jp, "w"), indent=1)
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")
    print("wrote", zp)

ens_test = {k: sum(P_TEST[m][k] for m in members) / len(members) for k in TEST_IDS}
uni_test = {k: sum(P_TEST[m][k] for m in uniform) / len(uniform) for k in TEST_IDS}
write(ens_test, BIAS if USE_BIAS else None, "ensemble")
write(uni_test, None, "uniform")
json.dump(dict(runs=names, members=[names[m] for m in members], dev_greedy=best, dev_bias_tuned=f_b,
               honest=[h0, h1], used_bias=bool(USE_BIAS), bias=BIAS.tolist()),
          open(OUT / "ensemble_results.json", "w"), indent=1)